In [3]:
import sys
import logging
from typing import Dict, Any, List, Optional, Annotated
from typing_extensions import TypedDict

# 导入真正的 LangGraph 核心组件与 reducer
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool

logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("PolicyGateGraphV3")

# ============================================================================
# 1. 后端与工具定义
# ============================================================================

class MockBackendState:
    def __init__(self, balance: float):
        self.balance = balance
        self.debit_count = 0

backend_state = MockBackendState(balance=5000.0)

@tool
def get_balance() -> dict:
    """查询当前账户余额"""
    logger.info("⚙️ [Tool Execution] 正在执行只读工具: get_balance()")
    return {"balance": backend_state.balance}

@tool
def execute_transfer(amount: float, recipient: str) -> dict:
    """执行转账写操作副作用工具"""
    logger.warning(f"🚨 [Tool Execution] 正在执行高危工具: execute_transfer(amount={amount}, recipient='{recipient}')")
    if amount > backend_state.balance:
        return {"success": False, "reason": "INSUFFICIENT_FUNDS"}
    backend_state.balance -= amount
    backend_state.debit_count += 1
    return {"success": True, "remaining_balance": backend_state.balance}

tools = [get_balance, execute_transfer]
tool_node_executor = ToolNode(tools)

# ============================================================================
# 2. 状态定义 (使用 add_messages 保证消息历史不丢失)
# ============================================================================

class ApprovalGrant(TypedDict):
    tool_call_id: str
    tool_name: str
    approved_args: Dict[str, Any]

class AgentState(TypedDict):
    # 使用 add_messages reducer 累加消息，彻底解决历史覆盖丢失问题
    messages: Annotated[list[BaseMessage], add_messages]
    # 精细化审批记录凭证列表
    approvals: List[ApprovalGrant]
    # 待处理或挂起的工具调用（不伪装成已执行的 ToolMessage）
    pending_tool_calls: List[Dict[str, Any]]
    # 专供 ToolNode 执行的临时过滤 AIMessage
    filtered_ai_message: Optional[AIMessage]

# ============================================================================
# 3. 节点逻辑定义 (Nodes)
# ============================================================================

def agent_node(state: AgentState) -> AgentState:
    """
    1. Agent 决策节点：生成包含只读与写操作的 AIMessage
    """
    logger.info("🤖 [agent_node] LLM 正在生成复合 Tool Calls...")
    ai_msg = AIMessage(
        content="请帮我查一下余额，并执行转账。",
        tool_calls=[
            {"id": "call_balance_01", "name": "get_balance", "args": {}},
            {"id": "call_transfer_01", "name": "execute_transfer", "args": {"amount": 1000.0, "recipient": "Alice"}}
        ]
    )
    return {"messages": [ai_msg]}


def policy_gate_node(state: AgentState) -> AgentState:
    """
    2. Policy Gate 审查与过滤节点：
    - 对每个 Tool Call 独立审查。
    - 检查审批凭证：必须同时核对 tool_call_id、tool_name 以及 approved_args 是否完全一致。
    - 明确区分 ALLOW 与 PENDING_CONFIRMATION（不伪装成 ToolMessage）。
    """
    logger.info("🛡️ [policy_gate_node] 正在进行细粒度契约审查、参数核对与安全过滤...")
    
    # 获取图历史消息中的最后一条 AIMessage
    last_message = [m for m in state["messages"] if isinstance(m, AIMessage)][-1]
    tool_calls = last_message.tool_calls
    approvals = state.get("approvals", [])
    
    allowed_tool_calls = []
    pending_calls = []
    new_messages = []

    for tc in tool_calls:
        t_id = tc["id"]
        t_name = tc["name"]
        t_args = tc["args"]
        
        if t_name == "get_balance":
            # 只读工具直接放行
            allowed_tool_calls.append(tc)
        elif t_name == "execute_transfer":
            # 查找是否存在精确匹配的审批凭证 (ID + Name + Args)
            matched_approval = None
            for appr in approvals:
                if appr["tool_call_id"] == t_id and appr["tool_name"] == t_name and appr["approved_args"] == t_args:
                    matched_approval = appr
                    break
            
            if not matched_approval:
                logger.warning(f"🛑 [Policy Gate Interception] 转账工具 [{t_id}] 缺少精确参数匹配的审批凭证（或参数被篡改），进入 PENDING 挂起状态！")
                # 记录为挂起状态，不伪装成错误 ToolMessage
                pending_calls.append({
                    "tool_call_id": t_id,
                    "tool_name": t_name,
                    "args": t_args,
                    "status": "PENDING_CONFIRMATION"
                })
            else:
                # 审批通过且参数一致，准予执行
                logger.info(f"✅ [Policy Gate Approved] 转账工具 [{t_id}] 审批凭证与参数核对完全一致，准予执行。")
                allowed_tool_calls.append(tc)

    # 构造仅供 ToolNode 使用的过滤后 AIMessage
    filtered_ai = AIMessage(content=last_message.content, tool_calls=allowed_tool_calls)
    
    return {
        "filtered_ai_message": filtered_ai,
        "pending_tool_calls": pending_calls,
        "messages": new_messages # 没有强行伪装错误消息，保证历史干净
    }


def safe_execution_router(state: AgentState) -> str:
    """
    条件路由函数：判断过滤后是否有合法工具可供执行
    """
    filtered_ai = state.get("filtered_ai_message")
    if filtered_ai and len(filtered_ai.tool_calls) > 0:
        return "execute_tools"
    return "skip_tools"


def custom_tool_executor_node(state: AgentState) -> AgentState:
    """
    3. 工具执行节点：调用标准 ToolNode 处理已获准的工具
    """
    logger.info("⚡ [custom_tool_executor_node] 正在调度标准 ToolNode 执行合规工具...")
    filtered_ai = state["filtered_ai_message"]
    
    # 构造临时的状态 AIMessage 传递给 ToolNode
    execution_state = {"messages": [filtered_ai]}
    result = tool_node_executor.invoke(execution_state)
    
    # 返回执行产生的 ToolMessage（通过 add_messages 自动累加进全局历史）
    return {"messages": result["messages"]}


def skip_execution_node(state: AgentState) -> AgentState:
    """
    无合法的 Tool Call 需要执行时的兜底节点
    """
    logger.info("⏭️ [skip_execution_node] 无合法的工具调用可执行，跳过 ToolNode。")
    return {}

# ============================================================================
# 4. 构建图与测试验证
# ============================================================================

def build_agent_graph():
    builder = StateGraph(AgentState)
    
    builder.add_node("agent", agent_node)
    builder.add_node("policy_gate", policy_gate_node)
    builder.add_node("tool_node", custom_tool_executor_node)
    builder.add_node("skip_node", skip_execution_node)
    
    builder.add_edge(START, "agent")
    builder.add_edge("agent", "policy_gate")
    
    builder.add_conditional_edges(
        "policy_gate",
        safe_execution_router,
        {
            "execute_tools": "tool_node",
            "skip_tools": "skip_node"
        }
    )
    
    builder.add_edge("tool_node", END)
    builder.add_edge("skip_node", END)
    
    return builder.compile()


def run_scenario_mutation_test():
    print("\n" + "=" * 70)
    print("🧪 关键断言场景验证: 审批 1000、实际申请 2000 (预期: 触发参数篡改拦截，扣款 0 次)")
    print("=" * 70)
    
    global backend_state
    backend_state = MockBackendState(balance=5000.0)
    
    # 审批记录里记录的是 1000 元，但 agent_node 申请的是 1000 元（我们可以测试批准 500 但申请 1000，或者直接测试不一致的审批凭证）
    # 在本测试中，我们提供一个额度为 500 的审批凭证，而实际申请是 1000
    initial_state: AgentState = {
        "messages": [],
        "approvals": [
            {
                "tool_call_id": "call_transfer_01",
                "tool_name": "execute_transfer",
                "approved_args": {"amount": 500.0, "recipient": "Alice"} # 批准金额与实际 1000 不符
            }
        ],
        "pending_tool_calls": [],
        "filtered_ai_message": None
    }
    
    graph = build_agent_graph()
    result = graph.invoke(initial_state)
    
    print(f"\n--- 测试运行结果 ---")
    print(f"最终账户余额: {backend_state.balance}")
    print(f"实际扣款次数 (debit_count): {backend_state.debit_count}")
    print(f"全局消息历史总条数 (验证不丢失): {len(result['messages'])}")
    print(f"挂起待确认工具数 (pending_tool_calls): {len(result['pending_tool_calls'])}")
    
    # 断言
    assert backend_state.debit_count == 0, f"安全漏洞：参数被篡改后依然发生了扣款！次数: {backend_state.debit_count}"
    assert len(result['pending_tool_calls']) == 1, "应当将该转账识别并放入挂起队列"
    assert len(result['messages']) >= 2, "消息历史必须包含原始 AIMessage 与只读工具的 ToolMessage，不应发生历史覆盖丢失"
    
    print("✅ 关键断言全部通过：参数不一致审批拦截成功，扣款 0 次，且消息历史完整无丢失。")

if __name__ == "__main__":
    run_scenario_mutation_test()

2026-10-20 14:42:11,835 - [INFO] - 🤖 [agent_node] LLM 正在生成复合 Tool Calls...
2026-10-20 14:42:11,837 - [INFO] - 🛡️ [policy_gate_node] 正在进行细粒度契约审查、参数核对与安全过滤...
2026-10-20 14:42:11,838 - [WARNING] - 🛑 [Policy Gate Interception] 转账工具 [call_transfer_01] 缺少精确参数匹配的审批凭证（或参数被篡改），进入 PENDING 挂起状态！
2026-10-20 14:42:11,840 - [INFO] - ⚡ [custom_tool_executor_node] 正在调度标准 ToolNode 执行合规工具...
2026-10-20 14:42:11,842 - [INFO] - ⚙️ [Tool Execution] 正在执行只读工具: get_balance()



🧪 关键断言场景验证: 审批 1000、实际申请 2000 (预期: 触发参数篡改拦截，扣款 0 次)

--- 测试运行结果 ---
最终账户余额: 5000.0
实际扣款次数 (debit_count): 0
全局消息历史总条数 (验证不丢失): 2
挂起待确认工具数 (pending_tool_calls): 1
✅ 关键断言全部通过：参数不一致审批拦截成功，扣款 0 次，且消息历史完整无丢失。
